# E: Drink-Drive

There are no hints and no answers in this lab sheet. Towards the end of the session, we will discuss a solution together - so you can compare yours with the one I will present.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_val_score

from sklearn.model_selection import GridSearchCV

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.preprocessing import (
    MinMaxScaler,
    RobustScaler,
    StandardScaler,
    OrdinalEncoder,
)


In [2]:
rng = np.random.RandomState(2)

## The business problem

Your friend has an idea for an app. At the close of an evening's socialising, users can enter a few details into an on-screen form and the app will advise them whether they are safe to drive home or whether they are over the drink-drive limit. Your friend will take care of building the app architecture. He has asked you to train a model that can make under-the-limit/over-the-limit predictions. Accuracy is essential. If the model can explain its predictions, so much the better.

## Performance measures

You realise that this is a binary classification problem and that accuracy is the main metric to use in the evaluation. It makes sense to compare against a baseline - a dummy classifier that predicts the majority-class of the training examples. 

## Dataset

You speak to your lecturer about this and he provides you with a CSV file (`drink_drive.csv`). This file was collected by two students during their PhD in Trinity. The students bought a large box of breathalyser devices. Over several evenings, they visited Dublin pubs and asked customers to be part of a dataset. Each row contains the responses of a customer to various questions, which become the features of the dataset. After responding to the questions, the customer blew into a breathalyser to determine whether they were under or over the drink-drive limit - "No" or "Yes" respectively. This is the 'ground truth'. It becomes the target value.

Each row in the CSV file contains 9 pieces of data, which your lecturer explains as follows:
- age_yrs: The person's age.
- height_cm: Their height.
- weight_kg: Their weight
- duration_mins: How long they've been drinking today (the time between the first sip of the first drink and the most recent sip of the most recent drink).
- elapsed_mins: How long since their last sip.
- sex: Biological sex.
- last_meal: the nature of their most recent meal, e.g. Snack, Lunch or Full (a full meal - a dinner).

(In truth, this dataset is too small for Machine Learning - fewer than 100 examples. You cannot do ML on this size of dataset. We're using it for this lab because it's fun and throws up several interesting issues. The two students with their supervisor and me did not care about learning an accurate predictive model. We were using it in our research on XAI. We were able to demonstrate an interesting new way of explaining a model's predictions - and, for this, the small dataset was just about adequate.)

In [3]:
import os

if 'google.colab' in str(get_ipython()):
    from google.colab import drive

    drive.mount('/content/drive')
    base_dir = './drive/My Drive/Colab Notebooks/'  # You may need to change this, depending on where your notebooks are on Google Drive
else:
    base_dir = '.'
dataset_dir = os.path.join(base_dir, 'datasets')

In [4]:
df: pd.DataFrame = pd.read_csv(os.path.join(dataset_dir, 'drink_drive.csv'))

Now you're on your own. You know the steps: take a cheeky look, clean anything that's invalid, split, EDA, Feature Engineering, Preprocessing, Model Selection and, after any necessary iteration, finally Error Estimation. To speed things along, you can skip EDA. But, include some ensembles this time.

## Take a cheeky look

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   age_yrs        76 non-null     int64  
 1   height_cm      76 non-null     int64  
 2   weight_kg      76 non-null     int64  
 3   duration_mins  76 non-null     object 
 4   elapsed_mins   76 non-null     object 
 5   sex            76 non-null     object 
 6   last_meal      65 non-null     object 
 7   units          76 non-null     float64
 8   over_limit     76 non-null     object 
dtypes: float64(1), int64(3), object(5)
memory usage: 5.5+ KB


In [6]:
df.head()

,age_yrs,height_cm,weight_kg,duration_mins,elapsed_mins,sex,last_meal,units,over_limit
0,40,170,75,?,?,Male,Lunch,0.0,No
1,26,177,76,60,10,Male,Full,2.9,No
2,24,160,60,60,10,Female,Full,2.6,No
3,29,160,63,90,10,Female,Full,1.2,No
4,23,182,63,120,10,Male,Full,5.2,No


In [7]:
df.describe(include='all')

,age_yrs,height_cm,weight_kg,duration_mins,elapsed_mins,sex,last_meal,units,over_limit
count,76.000000,76.000000,76.000000,76,76,76,65,76.000000,76
unique,NaN,NaN,NaN,16,9,2,4,NaN,2
top,NaN,NaN,NaN,120,10,Male,Full,NaN,No
freq,NaN,NaN,NaN,15,61,60,33,NaN,46
mean,22.657895,176.644737,71.486842,NaN,NaN,NaN,NaN,8.632895,NaN
std,5.627439,8.453329,11.474602,NaN,NaN,NaN,NaN,5.775567,NaN
min,18.000000,157.000000,47.000000,NaN,NaN,NaN,NaN,0.000000,NaN
25%,19.000000,172.000000,63.000000,NaN,NaN,NaN,NaN,4.275000,NaN
50%,21.000000,177.000000,72.000000,NaN,NaN,NaN,NaN,8.400000,NaN
75%,23.000000,182.000000,79.000000,NaN,NaN,NaN,NaN,12.100000,NaN


In [8]:
df['last_meal'].unique(), df['duration_mins'].unique(), df['elapsed_mins'].unique()

(array(['Lunch', 'Full', 'Snack', '?', nan], dtype=object),
 array(['?', '60', '90', '120', '150', '240', '30', '270', '180', '330',
        '435', '325', '300', '360', '315', '5'], dtype=object),
 array(['?', '10', '30', '5', '60', '40', '120', '15', '180'], dtype=object))

Notes:

- Null `last_meal` values -> could replace with mode?
- Could encode values for `last_meal` with ordinal encoder seeing as amount is increasing
- `last_meal` & `duration_mins` & `elapsed_mins` includes a "?" value which should be treated as null
- `duration_mins` and `elapsed_mins` should be int
- 31.2 units is a potential lie

## Cleanup anything that is simply invalid

In [9]:
# Clean up invalids in last_meal
df.loc[(df['last_meal'].isna()) | (df['last_meal'] == '?'), 'last_meal'] = (
    df['last_meal'].mode().iloc[0]
)

In [10]:
# Clean up invalids in duration_mins and elapsed_mins
df.loc[df['elapsed_mins'] == '?', 'elapsed_mins'] = df['elapsed_mins'].mode().iloc[0]
df.loc[df['duration_mins'] == '?', 'duration_mins'] = df['duration_mins'].mode().iloc[0]

In [11]:
# Convert into correct data types
df['duration_mins'] = df['duration_mins'].astype('int64')
df['elapsed_mins'] = df['elapsed_mins'].astype('int64')

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76 entries, 0 to 75
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   age_yrs        76 non-null     int64  
 1   height_cm      76 non-null     int64  
 2   weight_kg      76 non-null     int64  
 3   duration_mins  76 non-null     int64  
 4   elapsed_mins   76 non-null     int64  
 5   sex            76 non-null     object 
 6   last_meal      76 non-null     object 
 7   units          76 non-null     float64
 8   over_limit     76 non-null     object 
dtypes: float64(1), int64(5), object(3)
memory usage: 5.5+ KB


## Split into training set and test set

In [13]:
numeric_features = [
    'age_yrs',
    'height_cm',
    'weight_kg',
    'duration_mins',
    'elapsed_mins',
    'units',
]
nominal_features = ['sex', 'last_meal']

X = df[numeric_features + nominal_features]
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df.over_limit)

In [14]:
label_encoder.inverse_transform([0, 1])

array(['No', 'Yes'], dtype=object)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=df.over_limit, random_state=rng
)

In [16]:
y_train.sum() / y_train.shape[0], y_test.sum() / y_test.shape[0]

(np.float64(0.40350877192982454), np.float64(0.3684210526315789))

##  Exploratory Data Analysis

Skip this step. Otherwise, you won't finish during the lab session.

## Feature Engineering

Don't spend too long on this step - we practised it in a previous lab. 

## Preprocess the Data

In [17]:
preprocessor = ColumnTransformer(
    [
        (
            'num',
            Pipeline(
                [
                    ('scaler', None),
                ]
            ),
            numeric_features,
        ),
        ('sex', Pipeline([('encoder', OneHotEncoder(drop='if_binary'))]), ['sex']),
        ('meal', Pipeline([('encoder', None)]), ['last_meal']),
    ],
    remainder='drop',
)

## Model selection

In [18]:
dummy = DummyClassifier()
dummy.fit(X_train, y_train)

np.mean(cross_val_score(dummy, X_train, y_train, scoring='accuracy', cv=10))

np.float64(0.5966666666666665)

In [19]:
def grid_search(X_train, y_train, preprocessor, predictor, param_grid, cv, metric):
    model = Pipeline([('preprocessor', preprocessor), ('predictor', predictor)])

    gs = GridSearchCV(
        model, param_grid, scoring=metric, cv=cv, n_jobs=-1, error_score='raise'
    )

    gs.fit(X_train, y_train)

    return gs

In [20]:
param_grid = {'preprocessor__meal__encoder': [OneHotEncoder(), OrdinalEncoder()]}

In [21]:
dt_gs = grid_search(
    X_train,
    y_train,
    preprocessor=preprocessor,
    predictor=DecisionTreeClassifier(random_state=rng),
    param_grid={
        **param_grid,
        'predictor__max_depth': range(1, 11),
    },
    cv=10,
    metric='accuracy',
)

dt_gs.best_params_, dt_gs.best_score_

({'predictor__max_depth': 1, 'preprocessor__meal__encoder': OneHotEncoder()},
 np.float64(0.7666666666666666))

In [ ]:
knn_gs = grid_search(
    X_train, 
    y_train,
    preprocessor=preprocessor,
    predictor=De
)

TypeError: grid_search() missing 5 required positional arguments: 'preprocessor', 'predictor', 'param_grid', 'cv', and 'metric'

## Error Estimation - evaluate on the the test set

- The invalid ? for `elapsed_mins` & `duration_mins` were because they hadn't been drinking, so deleted
- Replaced no `last_meal` with None
- Need to look at which ones are null and try figure out why null
- Look at outliers and decide if they are outliers
- Could use a box whisker plot to get an idea of if the new feature is indicative in classifier, if little overlap then possibly good
- Keep the cv value low because the dataset is so small
- Leave-one-out classifier could work well, look into it
- Should be replacing the NaN with an imputer instead of doing it manually

CA Questions:

- How long: just over 2 weeks
- How scored: does it run, leakage = lost marks, have comments and paragraphs in the notebook, engage with the data (talk about what each bit means in terms of the data)
- 